In [ ]:
# !pip3 install --upgrade 'google-cloud-bigquery[bqstorage,pandas]'

In [28]:
import pandas as pd
from datetime import datetime
import json
import requests


categories = [
    ['17', 'processor'],
    ['12', 'motherboard'],
    ['3', 'casing'],
    ['24', 'vga'],
    ['9', 'lcd'],
    ['11', 'memory-ram'],
    ['101', 'solid-state-drive'],
    ['6', 'hard-disk']
]

def ingest_enterkomputer(cat_id, cat):
    
    page_counter = 1
    status = True

    while status:
        url = 'https://www.enterkomputer.com/jeanne/v2/product-list'
        headers = {
            'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:124.0) Gecko/20100101 Firefox/124.0',
            'Accept': 'application/json',
            'Accept-Language': 'en-US,en;q=0.5',
            'Accept-Encoding': 'gzip, deflate, br',
            'Content-Type': 'application/json',
            'X-Requested-With': 'XMLHttpRequest',
            'Origin': 'https://www.enterkomputer.com',
            'DNT': '1',
            'Sec-GPC': '1',
            'Connection': 'keep-alive',
            'Referer': f'https://www.enterkomputer.com/category/{cat_id}/{cat}',
            'Cookie': 'ess=a63e8c792788789ed3ca4488338282c2666be8f4; csrf_cookie_name=2221d451c939048e8e79b219af64d8ea',
            'Sec-Fetch-Dest': 'empty',
            'Sec-Fetch-Mode': 'cors',
            'Sec-Fetch-Site': 'same-origin',
            'TE': 'trailers'
        }

        data = {
            "KCODE": cat_id,
            "SCODE": "all",
            "BCODE": "all",
            "BNAME": "",
            "MORDR": "default",
            "MSTGE": "mapping",
            "MKYWD": "",
            "MTAGS": "",
            "MSGMN": "category",
            "MPAGE": page_counter,
            "token": "U2FsdGVkX1-E55sT1JEmUtTtgjHvzgK98PZU8pKsTjQf8t2cV6U0Rrrd5ijzmdtRiKOvKb944B267vLzsZdvag",
            "signature": "54f641b51cc26b51894b06dbdb55b4b3"
        }

        json_response = json.loads(requests.post(url, headers=headers, json=data).text)
        print(cat, page_counter, json_response['status'])

        page_counter += 1
        status = json_response['status']
        
        if status:
            products = json_response['result'][0]['PPRNT'][0]['PCHLD']
            for p1 in products:
                for p2 in p1['PLIST']:
                    product_list.append(p2)

In [29]:
product_list = []

for cat in categories:
    ingest_enterkomputer(cat[0], cat[1])

processor 1 True
processor 2 True
processor 3 True
processor 4 True
processor 5 False
motherboard 1 True
motherboard 2 True
motherboard 3 True
motherboard 4 True
motherboard 5 True
motherboard 6 True
motherboard 7 True
motherboard 8 True
motherboard 9 True
motherboard 10 False
casing 1 True
casing 2 True
casing 3 True
casing 4 True
casing 5 True
casing 6 True
casing 7 True
casing 8 True
casing 9 True
casing 10 True
casing 11 True
casing 12 True
casing 13 True
casing 14 True
casing 15 True
casing 16 True
casing 17 False
vga 1 True
vga 2 True
vga 3 True
vga 4 True
vga 5 True
vga 6 True
vga 7 True
vga 8 True
vga 9 True
vga 10 True
vga 11 False
lcd 1 True
lcd 2 True
lcd 3 True
lcd 4 True
lcd 5 True
lcd 6 True
lcd 7 True
lcd 8 True
lcd 9 True
lcd 10 True
lcd 11 True
lcd 12 True
lcd 13 True
lcd 14 False
memory-ram 1 True
memory-ram 2 True
memory-ram 3 True
memory-ram 4 True
memory-ram 5 True
memory-ram 6 True
memory-ram 7 True
memory-ram 8 True
memory-ram 9 True
memory-ram 10 True
memory-ram

In [30]:
df = pd.DataFrame(product_list)
df = df.astype(str)
df['inserted_at'] = pd.Timestamp(datetime.now())

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2485 entries, 0 to 2484
Data columns (total 22 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   KCODE        2485 non-null   object        
 1   KNAME        2485 non-null   object        
 2   SCODE        2485 non-null   object        
 3   BCODE        2485 non-null   object        
 4   PCODE        2485 non-null   object        
 5   PNAME        2485 non-null   object        
 6   PPRCZ        2485 non-null   object        
 7   PWGHT        2485 non-null   object        
 8   PQTTY        2485 non-null   object        
 9   PTYPE        2485 non-null   object        
 10  PSTTS        2485 non-null   object        
 11  PBEST        2485 non-null   object        
 12  PDISP        2485 non-null   object        
 13  PDTLS        2485 non-null   object        
 14  MKYWD        2485 non-null   object        
 15  MTAGS        2485 non-null   object        
 16  PTAGS 

## To BQ

In [33]:
from google.oauth2 import service_account
import pandas_gbq

credentials = service_account.Credentials.from_service_account_file(
    '/home/al/projects/creds/ichsanul-dev-cc6f799c9121.json', scopes=['https://www.googleapis.com/auth/cloud-platform']
)
# Replace with your own GCP credentials
project_id = "ichsanul-dev"
dataset_id = "de_zoomcamp"
table_id = "de_zoomcamp.enterkomputer"

pandas_gbq.to_gbq(dataframe=df, credentials=credentials, destination_table=table_id, if_exists='append')

100%|██████████| 1/1 [00:00<00:00, 17189.77it/s]


## To PG

In [ ]:
from sqlalchemy import create_engine
db_params = {
    "dbname": "postgres",
    "user": "postgres",
    "password": "postgres",
    "host": "localhost",  # Change to your database host
    "port": "5432"       # Change to your database port
}
db_url = f"postgresql://{db_params['user']}:{db_params['password']}@{db_params['host']}:{db_params['port']}/{db_params['dbname']}"
engine = create_engine(db_url)

In [ ]:
# df.to_sql(name='enterkomputer', con=engine, schema='personal', if_exists='append', method='multi', index=False)